# Ligands

The `ligands` module represents **small molecules** for structure prediction, docking, and visualization. This notebook walks through the core workflow:

1. Build a ligand from a SMILES string
2. Generate 3D conformers and export to SDF
3. Load cofactors by CCD code
4. Visualize a molecule in 3D

Every cell runs as-is; the module ships with `rdkit` and `py3Dmol`.

## Build a ligand from SMILES

A `Fragment` is a single connected molecule. SMILES are validated on construction and stored in canonical form.

In [1]:
from proto_tools.entities.ligands import Fragment, Ligands

aspirin = Fragment(smiles="CC(=O)OC1=CC=CC=C1C(=O)O", name="Aspirin")
print(f"Canonical SMILES: {aspirin.smiles}")

Canonical SMILES: CC(=O)Oc1ccccc1C(=O)O


## Generate 3D conformers

`generate_conformers` embeds and UFF-minimizes 3D structures with RDKit (ETKDGv3). Near-duplicate conformers are pruned, so the count returned can be lower than requested.

In [2]:
aspirin.generate_conformers(num_conformers=5)
print(f"Generated {len(aspirin.conformers)} conformers")

Generated 2 conformers


## Collections and SDF export

`Ligands` is a collection of fragments. Use the factories (`from_smiles`, `from_file`, `from_ccd_codes`, `from_mols`) to load from common sources, then `to_sdf` to export for external tools. Conformers are required before export.

In [3]:
import tempfile
from pathlib import Path

# Ibuprofen, loaded as a single-fragment collection
drug = Ligands.from_smiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O")
drug.generate_conformers(num_conformers=1)

with tempfile.TemporaryDirectory() as tmp:
    sdf_path = Path(tmp) / "ibuprofen.sdf"
    drug.to_sdf(sdf_path)
    print(f"Wrote {sdf_path.name} ({sdf_path.stat().st_size} bytes)")

Wrote ibuprofen.sdf (2816 bytes)


## Load cofactors by CCD code

Cofactors and ions can be loaded by their wwPDB Chemical Component Dictionary (CCD) code; each code is resolved to SMILES from the offline CCD database.

In [4]:
cofactors = Ligands.from_ccd_codes(["ATP", "MG"])
print(f"{len(cofactors)} fragments")
for fragment in cofactors:
    print(f"  {fragment.ccd_code}: {fragment.smiles}")

2 fragments
  ATP: Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@@](=O)(O)O[P@](=O)(O)OP(=O)(O)O)[C@@H](O)[C@H]1O
  MG: [Mg+2]


## Visualize in 3D

`visualize()` renders the molecule with py3Dmol. It auto-generates a conformer if none exist, so it works on a freshly loaded ligand.

In [5]:
# Caffeine
caffeine = Ligands.from_smiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
caffeine.visualize(style="stick")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.